# 01 - Limpieza: comercio exterior 2016-2025 (INDEC comex)

Carga las bases anuales de **importaciones** y **exportaciones** por producto (NCM) y país, valida **cada año**
contra los totales de control que publica INDEC, asigna cada producto a una **rama industrial**
(`src/ramas.py`) y guarda tablas agregadas chicas en `data/processed/`.

## Cosas a tener presentes
- **CIF vs FOB.** 2016-2018 publican solo valor CIF (incluye flete y seguro); desde 2019 también FOB. Para tener una
  serie comparable todos los años, las importaciones se miden en **CIF** y en **toneladas** (esta última no
  depende de los precios: sirve para distinguir "importamos más cosas" de "las mismas cosas más baratas").
- **Valores nominales** en dólares corrientes, sin ajustar por inflación internacional.
- **Exportaciones confidenciales.** Desde 2018, INDEC oculta productos con pocos exportadores (NCM `99999999`,
  ~5% del valor): quedan sin rama y se muestran aparte.
- Los archivos crudos pesan ~100 MB y no se suben al repositorio: se bajan con `src/download_comex.py`.

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import analysis as A
import loaders as L
from ramas import RAMA_LABELS, TOTAL

RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(exist_ok=True)

In [2]:
import subprocess

# idempotente: si los archivos ya están, no baja nada
print(subprocess.run([sys.executable, str(PROJECT_ROOT / "src" / "download_comex.py")],
                     capture_output=True, text=True).stdout[-400:])

2019: ya estaba
imports 2020: ya estaba
imports 2021: ya estaba
imports 2022: ya estaba
imports 2023: ya estaba
imports 2024: ya estaba
imports 2025: ya estaba
exports 2016: ya estaba
exports 2017: ya estaba
exports 2018: ya estaba
exports 2019: ya estaba
exports 2020: ya estaba
exports 2021: ya estaba
exports 2022: ya estaba
exports 2023: ya estaba
exports 2024: ya estaba
exports 2025: ya estaba



## Validación año por año contra los totales de control

Para cada año se compara la cantidad de registros y el valor total contra lo que publica INDEC. Si algo no
coincide se cortó una lectura (encoding, separador decimal, filas truncadas).

In [3]:
AÑOS = range(2016, 2026)
imps, exps, control = [], [], []

for y in AÑOS:
    imp, ex = L.load_importaciones(y), L.load_exportaciones(y)
    ci, ce = L.control_importaciones(y), L.control_exportaciones(y)

    assert len(imp) == ci["registros"] and abs(imp["cif_usd"].sum() - ci["cif_usd"]) < 1, f"importaciones {y}"
    assert len(ex) == ce["registros"] and abs(ex["fob_usd"].sum() - ce["fob_usd"]) < 1, f"exportaciones {y}"
    control.append(dict(anio=y, registros_imp=len(imp), cif_total=imp["cif_usd"].sum(), coincide_imp=True,
                        registros_exp=len(ex), fob_total=ex["fob_usd"].sum(), coincide_exp=True))

    imps.append(imp.assign(rama=imp["rama"].fillna("NO_IND")).groupby(["anio", "rama", "pais"], as_index=False)
                .agg(cif_usd=("cif_usd", "sum"), fob_usd=("fob_usd", lambda s: s.sum(min_count=1)),
                     peso_neto_kg=("peso_neto_kg", "sum")))
    ex["rama"] = ex["rama"].fillna(ex["ncm"].map(lambda n: "CONFIDENCIAL" if n == "99999999" else "NO_IND"))
    exps.append(ex.groupby(["anio", "rama"], as_index=False).agg(fob_usd=("fob_usd", "sum"), peso_neto_kg=("peso_neto_kg", "sum")))

control = pd.DataFrame(control)
print("Los 10 años de importaciones y de exportaciones coinciden con los totales de control de INDEC.")
control.assign(cif_total_mm=lambda d: (d.cif_total / 1e6).round(0), fob_total_mm=lambda d: (d.fob_total / 1e6).round(0))[
    ["anio", "registros_imp", "cif_total_mm", "registros_exp", "fob_total_mm"]]

Los 10 años de importaciones y de exportaciones coinciden con los totales de control de INDEC.


,anio,registros_imp,cif_total_mm,registros_exp,fob_total_mm
0,2016,69685,55852.0,39412,57909.0
1,2017,73085,66938.0,39648,58645.0
2,2018,73567,65483.0,44414,61782.0
3,2019,70979,49125.0,45915,65115.0
4,2020,66896,42356.0,41144,54884.0
5,2021,69935,63184.0,42250,77934.0
6,2022,69732,81740.0,41558,88703.0
7,2023,66408,73816.0,39145,66956.0
8,2024,70724,60776.0,40578,79703.0
9,2025,76148,75791.0,41057,87111.0


## Cobertura del mapeo NCM → rama

Lo que queda sin rama debería ser *no manufactura*: agro y ganadería sin procesar (capítulos 1-14), minerales (26),
petróleo crudo, gas natural, carbón y electricidad (partes del capítulo 27 que no son refinación), oro y plata sin
trabajar (capítulo 71) y códigos especiales (98-99). Se excluyen del análisis por rama pero se cuentan en los
totales de importaciones.

In [4]:
imp_all = pd.concat(imps)
tot = imp_all.groupby("anio").cif_usd.sum()
no_ind = imp_all[imp_all.rama == "NO_IND"].groupby("anio").cif_usd.sum()
(no_ind / tot * 100).round(1).rename("% del CIF sin rama (no manufactura)").to_frame().T

anio,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
% del CIF sin rama (no manufactura),8.6,9.5,12.6,11.9,13.0,13.5,13.9,16.7,12.9,9.5


In [5]:
imp_all.to_csv(PROCESSED / "importaciones_rama_pais_anio.csv", index=False)
pd.concat(exps).to_csv(PROCESSED / "exportaciones_rama_anio.csv", index=False)
control.to_csv(PROCESSED / "control_comex.csv", index=False)

resumen = imp_all.groupby("anio").agg(imp_cif=("cif_usd", "sum"), imp_kg=("peso_neto_kg", "sum")).join(
    imp_all[imp_all.pais == "China"].groupby("anio").cif_usd.sum().rename("imp_china_cif")).join(
    pd.concat(exps).groupby("anio").fob_usd.sum().rename("exp_fob")).reset_index()
resumen["pct_china"] = resumen["imp_china_cif"] / resumen["imp_cif"] * 100
resumen.to_csv(PROCESSED / "resumen_comercio_anual.csv", index=False)
resumen.assign(imp_cif=lambda d: (d.imp_cif / 1e9).round(1), exp_fob=lambda d: (d.exp_fob / 1e9).round(1),
               imp_kg=lambda d: (d.imp_kg / 1e9).round(1), pct_china=lambda d: d.pct_china.round(1)).drop(columns="imp_china_cif")

,anio,imp_cif,imp_kg,exp_fob,pct_china
0,2016,55.9,35.4,57.9,18.7
1,2017,66.9,40.0,58.6,18.4
2,2018,65.5,43.8,61.8,18.4
3,2019,49.1,32.0,65.1,18.8
4,2020,42.4,31.8,54.9,20.4
5,2021,63.2,40.8,77.9,21.4
6,2022,81.7,38.9,88.7,21.4
7,2023,73.8,39.5,67.0,19.7
8,2024,60.8,31.5,79.7,19.2
9,2025,75.8,32.8,87.1,23.7
